# Regional HAB forecasting with TabFM and LightGBM

Average the three class probabilities from TabFM and tuned LightGBM with
equal weights, then evaluate the severity forecast and an early warning policy.
Use the same labeled weeks and chronological splits as the standalone models.

Run [notebook 08](08_TabFM_Regional_HAB.ipynb) first to generate the TabFM forecasts.
See the [ensemble model summary](../TabFM_LightGBM_Ensemble_Development_Summary.md)
for setup, results, and the saved files.

## Load the regional data and align the weekly predictions

Verify that the saved TabFM forecasts use this dataset and cover each evaluation
week exactly once. Fit tuned LightGBM and CatBoost on the training part of each
fold. Keep class order fixed at 0, 1, and 2 before averaging probabilities.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

here = Path.cwd()
root = next(
    (path for path in (here, *here.parents)
     if (path / "notebooks" / "hab_tabfm.py").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Open this notebook from the project checkout.")
sys.path.insert(0, str(root / "notebooks"))
data_dir = root / "data" / "final"

from hab_tabfm import (
    PROB_COLUMNS, REPORT_WINDOWS, TABFM_WEIGHT, artifact_config, baseline_backtests,
    evaluate, fit_lightgbm, load_regional, load_tabfm, load_tabfm_forecasts,
    observed_forecast_weeks, probability_matrix, verify_tabfm_context,
)

raw, labeled, features = load_regional(data_dir)
observed = observed_forecast_weeks(labeled)
tabfm_dir = root / "models" / "tabfm"
if not (tabfm_dir / "config.json").is_file():
    raise FileNotFoundError("Run notebook 08 to create the local TabFM artifacts.")
tabfm_forecasts = load_tabfm_forecasts(tabfm_dir, data_dir, features, observed)
# Initialize PyTorch before the boosting libraries load their OpenMP runtimes.
tabfm_model = load_tabfm()
lightgbm_forecasts, catboost_forecasts = baseline_backtests(labeled, features)
if not lightgbm_forecasts.index.equals(tabfm_forecasts.index):
    raise ValueError("The two ensemble members must predict the same weeks in order.")
ensemble = TABFM_WEIGHT * tabfm_forecasts + (1 - TABFM_WEIGHT) * lightgbm_forecasts
forecasts = {
    "LightGBM": lightgbm_forecasts,
    "CatBoost": catboost_forecasts,
    "TabFM": tabfm_forecasts,
    "TabFM + LightGBM": ensemble,
}
print(f"Aligned {len(ensemble)} weekly forecasts from both ensemble members.")
display(ensemble.head().round(4))

/Users/arniskc/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights from local directory


Aligned 365 weekly forecasts from both ensemble members.


,p_class_0,p_class_1,p_class_2
target_week,,,
2017-01-02,0.0041,0.0250,0.9709
2017-01-09,0.0112,0.0853,0.9035
2017-01-16,0.0041,0.0276,0.9683
2017-01-23,0.0046,0.0274,0.9680
2017-01-30,0.0042,0.0337,0.9621


## Evaluation

The target is the next week's regional *K. brevis* severity. Evaluate only labeled
weeks. For each year from 2017 to 2024, use earlier years as training data and apply
a four-week gap before the first labeled target week. The 2020–2024 subset matches
notebook 07. For the 2025 test, train through 2024. The 2026 holdout is outside this run.

An **onset** is a target week at class 1 or 2 after the last observed class was 0.
A **severe miss** is class 2 called class 0. A **false alarm** is class 0 called
class 1 or 2. Onset recall is undefined in a window with zero onsets.
The cost scores add each false alarm to two or three times the severe-miss count.
Onset average precision ranks risk only on weeks whose last observed class was 0.

## Alert threshold comparison

Use the class with highest probability for ordinary severity forecasts. For early
warning, use P(class 1) + P(class 2) and predict class 0 below the cutoff. Otherwise,
pick the more probable of classes 1 and 2. CatBoost uses the published cutoff of 0.35.

For each year from 2020 onward, count CatBoost's false alarms on earlier forecast
years. Use only those earlier class-0 weeks to choose a candidate risk cutoff that
aims for the same count. Apply the cutoff to the next year. The realized false-alarm
counts can differ. This compares alert policies on the historical data.

In [2]:
summary, weekly, cutoffs = evaluate(observed, forecasts)
for window in REPORT_WINDOWS:
    print(window)
    display(summary.loc[window].round(3))

Historical comparison (2020–2024)


,weeks,onsets caught,onsets,onset recall,severe misses,false alarms,cost 2:1,cost 3:1,accuracy,balanced accuracy,macro F1,onset AP,log loss
model,,,,,,,,,,,,,
Persistence,156,0,13,0.000,2,12,16,18,0.763,0.674,0.677,NaN,NaN
LightGBM,156,2,13,0.154,2,23,27,29,0.660,0.583,0.586,0.141,0.847
LightGBM (matched alert budget),156,6,13,0.462,0,53,53,53,0.506,0.547,0.518,0.141,0.847
CatBoost (0.35),156,6,13,0.462,0,33,33,33,0.660,0.659,0.636,0.205,0.654
TabFM,156,0,13,0.000,4,9,17,21,0.750,0.635,0.627,0.154,0.704
TabFM (matched alert budget),156,6,13,0.462,0,33,33,33,0.654,0.653,0.617,0.154,0.704
TabFM + LightGBM,156,1,13,0.077,2,14,18,20,0.718,0.616,0.613,0.149,0.740
TabFM + LightGBM (matched alert budget),156,7,13,0.538,0,44,44,44,0.571,0.590,0.565,0.149,0.740


Extended backtest (2017–2024)


,weeks,onsets caught,onsets,onset recall,severe misses,false alarms,cost 2:1,cost 3:1,accuracy,balanced accuracy,macro F1,onset AP,log loss
model,,,,,,,,,,,,,
Persistence,313,0,25,0.00,5,25,35,40,0.786,0.704,0.704,NaN,NaN
LightGBM,313,6,25,0.24,6,38,50,56,0.722,0.654,0.651,0.230,0.704
CatBoost (0.35),313,13,25,0.52,1,63,65,66,0.684,0.686,0.654,0.298,0.620
TabFM,313,2,25,0.08,8,20,36,44,0.776,0.664,0.667,0.243,0.655
TabFM + LightGBM,313,3,25,0.12,6,27,39,45,0.760,0.668,0.669,0.259,0.642


Final test (2025)


,weeks,onsets caught,onsets,onset recall,severe misses,false alarms,cost 2:1,cost 3:1,accuracy,balanced accuracy,macro F1,onset AP,log loss
model,,,,,,,,,,,,,
Persistence,52,0,0,NaN,0,1,1,1,0.962,0.659,0.637,NaN,NaN
LightGBM,52,0,0,NaN,0,3,3,3,0.904,0.589,0.600,NaN,0.300
LightGBM (matched alert budget),52,0,0,NaN,0,7,7,7,0.827,0.559,0.562,NaN,0.300
CatBoost (0.35),52,0,0,NaN,0,4,4,4,0.904,0.637,0.626,NaN,0.309
TabFM,52,0,0,NaN,0,1,1,1,0.962,0.659,0.637,NaN,0.207
TabFM (matched alert budget),52,0,0,NaN,0,3,3,3,0.923,0.644,0.630,NaN,0.207
TabFM + LightGBM,52,0,0,NaN,0,2,2,2,0.923,0.596,0.604,NaN,0.246
TabFM + LightGBM (matched alert budget),52,0,0,NaN,0,4,4,4,0.885,0.581,0.596,NaN,0.246


## Save and verify the ensemble artifacts

Save the blend probabilities, metrics, and weekly decisions. Fit and save the
LightGBM model through 2024 in its native text format. Load that model and reuse
the saved TabFM context from notebook 08 to reproduce one 2025 blend forecast.

In [3]:
import json
from lightgbm import Booster

output_dir = root / "models" / "tabfm_lightgbm"
output_dir.mkdir(parents=True, exist_ok=True)
ensemble.to_parquet(output_dir / "ensemble_probabilities.parquet")
summary.to_csv(output_dir / "metrics.csv")
weekly.to_csv(output_dir / "weekly_decisions.csv")

final_train = labeled.loc[labeled["target_year"] < 2025]
final_lightgbm = fit_lightgbm(final_train, features)
lightgbm_path = output_dir / "lightgbm_through_2024.txt"
final_lightgbm.booster_.save_model(str(lightgbm_path))
config = {
    **artifact_config(data_dir, features),
    "weights": {"TabFM": TABFM_WEIGHT, "LightGBM": 1 - TABFM_WEIGHT},
    "tabfm_context": "../tabfm/tabfm_context_through_2024.parquet",
    "lightgbm_model": lightgbm_path.name,
    "matched_alert_cutoffs": cutoffs,
    "context_through_year": 2024,
}
config_path = output_dir / "config.json"
config_path.write_text(json.dumps(config, indent=2) + "\n")

saved = json.loads(config_path.read_text())
sample = labeled.loc[labeled["target_year"] == 2025].head(1)
tabfm_sample = verify_tabfm_context(tabfm_dir, tabfm_model, sample, tabfm_forecasts)
booster = Booster(model_file=str(output_dir / saved["lightgbm_model"]))
lightgbm_sample = probability_matrix(booster.predict(sample[saved["regional_core"]], num_threads=1))
reloaded = (
    saved["weights"]["TabFM"] * tabfm_sample
    + saved["weights"]["LightGBM"] * lightgbm_sample
)
np.testing.assert_allclose(reloaded, ensemble.loc[reloaded.index], atol=1e-5, rtol=1e-5)
display(reloaded[PROB_COLUMNS].round(6))
print("Saved and verified models/tabfm_lightgbm/")

,p_class_0,p_class_1,p_class_2
target_week,,,
2025-01-06,0.030814,0.069986,0.8992


Saved and verified models/tabfm_lightgbm/
